# FILE 2/3 — Luồng B, Chặng 2: Fine-tune B-CanTho → HCMC (seed = 1712)

Khoi tao tu model B-CanTho sinh ra o FILE 1, fine-tune tiep tren HCMC de tao ra
**model B cuoi cung** (seed 1712).

**Vi tri trong quy trinh:**
- FILE 1: COCO → CanTho (da xong)
- **FILE 2 (notebook nay):** B-CanTho → HCMC → model B cuoi cung
- FILE 3: train luồng A + danh gia + bang so sanh 3 seed

**BAT BUOC chay FILE 1 xong truoc.**

> **LUU Y QUAN TRONG NEU CHAY BANG TAI KHOAN DUOC CHIA SE (vi du tai khoan cua Tai):**
> Thu muc "Do an AI giao thong" duoc share se nam o muc **"Shared with me"**, KHONG nam trong
> "My Drive", nen duong dan `/content/drive/MyDrive/Do an AI giao thong/...` se KHONG tim thay.
>
> **Cach xu ly (chi lam 1 lan, khong can sua code):**
> 1. Mo Google Drive tren web, vao muc **"Shared with me"**.
> 2. Tim thu muc **"Do an AI giao thong"**, chuot phai → **Organize** → **Add shortcut**.
> 3. Chon vi tri **"My Drive"** (thu muc goc) → bam **Add**.
>
> Sau buoc nay, moi duong dan trong notebook se hoat dong binh thuong.
> Cell "Buoc 3" ben duoi co dong kiem tra ton tai, neu bao False thi lam lai buoc tren.


## Buoc 1: Cai dat thu vien

In [1]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.5 MB/s eta 0:00:00
Da cai dat xong ultralytics


## Buoc 2: Ket noi Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Buoc 3: Khai bao duong dan va cau hinh

In [3]:
import os

SEED = 1712

# ===== MODEL KHOI TAO: model B-CanTho tu FILE 1 =====
PRETRAINED_B_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_cantho_seed1712/weights/best.pt"

# ===== DATASET =====
DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"
LOCAL_DIR_HCMC = "/content/hcmc_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_B_FINAL = "finetune_hcmc_from_cantho_seed1712"

IMGSZ, BATCH, EPOCHS, PATIENCE, LR0 = 640, 16, 100, 20, 0.001

print("Kiem tra su ton tai:")
print("  Model B-CanTho (FILE 1):", os.path.exists(PRETRAINED_B_CANTHO))
print("  Dataset HCMC (Drive)   :", os.path.exists(DRIVE_DIR_HCMC))
if not os.path.exists(PRETRAINED_B_CANTHO):
    print("\n[LOI] Chua tim thay model B-CanTho. Hay chay xong FILE 1 truoc!")

Kiem tra su ton tai:
  Model B-CanTho (FILE 1): True
  Dataset HCMC (Drive)   : True


## Buoc 4: Copy dataset HCMC ve o cung local

In [4]:
import shutil, time

if os.path.exists(LOCAL_DIR_HCMC):
    shutil.rmtree(LOCAL_DIR_HCMC)
t0 = time.time()
try:
    shutil.copytree(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC)
    print(f"[OK] Da copy HCMC ve {LOCAL_DIR_HCMC} ({time.time()-t0:.1f}s)")
except shutil.Error as e:
    print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy.")

DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"
print("Yaml HCMC (local) ton tai:", os.path.exists(DATA_YAML_HCMC_LOCAL))

[OK] Da copy HCMC ve /content/hcmc_701515 (735.0s)
Yaml HCMC (local) ton tai: True


## Buoc 5: Huan luyen chang 2 — B-CanTho → HCMC (seed 1712)

Khoi tao tu model B-CanTho (KHONG phai tu COCO goc) — day chinh la diem tao nen tinh
"tuan tu" cua lo trinh B. Mat khoang 1-1.5 gio.

In [ ]:
from ultralytics import YOLO

model_b = YOLO(PRETRAINED_B_CANTHO)
results = model_b.train(
    data=DATA_YAML_HCMC_LOCAL,
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, seed=SEED,
    project=PROJECT_DIR, name=NAME_B_FINAL, exist_ok=True, verbose=True,
)
print("\nDa huan luyen xong chang 2 (B-CanTho -> HCMC).")
BEST_B_FINAL = os.path.join(PROJECT_DIR, NAME_B_FINAL, "weights", "best.pt")
print("Model B cuoi cung luu tai:", BEST_B_FINAL)
print("Ton tai:", os.path.exists(BEST_B_FINAL))
print("\n>>> DA XONG FILE 2. Tiep theo chay FILE 3.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.148 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0,

## Buoc 5.1: Tiếp tục huấn luyện từ checkpoint last.pt sau khi runtime bị ngắt

In [5]:
from ultralytics import YOLO
import os

LAST_PT = os.path.join(
    PROJECT_DIR,
    NAME_B_FINAL,
    "weights",
    "last.pt"
)

print("Checkpoint:", LAST_PT)
print("last.pt tồn tại:", os.path.exists(LAST_PT))

if not os.path.exists(LAST_PT):
    raise FileNotFoundError("Không tìm thấy last.pt")

model_b_resume = YOLO(LAST_PT)
results_resume = model_b_resume.train(resume=True)

BEST_B_FINAL = os.path.join(
    PROJECT_DIR,
    NAME_B_FINAL,
    "weights",
    "best.pt"
)

print("\nĐã tiếp tục huấn luyện File 2.")
print("best.pt tồn tại:", os.path.exists(BEST_B_FINAL))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Checkpoint: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_from_cantho_seed1712/weights/last.pt
last.pt tồn tại: True
Ultralytics 8.4.148 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=